# 00 · Setup, smoke test and T4 timing calibration (Phase 2)

**Runtime → Change runtime type → T4 GPU.** Then *Run all*. About 20–30 min of GPU time.

This notebook:
1. installs the official efficient-SID code at the audited commit, plus our `sidx` package;
2. answers open item **D3**: what does `autocast(dtype=float32)` do to bf16 tensors on this GPU?
3. runs `configs/jobs/phase2_t4_timing.yaml` through the resumable runner (outputs on Drive);
4. writes the **timing table** that recalibrates PROJECT_PLAN §5.3;
5. checks that `faiss-gpu-cu12` imports (needed only for ablation AB6).

If the session disconnects, rerun everything: finished samples are skipped.

In [1]:
# ==== SETTINGS (edit once) ====================================================================
CODE_SOURCE   = "github"                                      # "github" or "drive"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
REPO_SUBDIR   = ""                                            # project files are at the repo root
DRIVE_CODE    = "/content/drive/MyDrive/CS787/code/Project"   # used when CODE_SOURCE == "drive"
DRIVE_ROOT    = "/content/drive/MyDrive/CS787"                # outputs, checkpoints, tables, env logs
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
# If the GitHub repo is private: add a Colab secret named GH_TOKEN (key icon in the left bar)
# holding a fine-grained, read-only token for that repo.

NOTEBOOK = "00_setup_smoke"

In [2]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU (then rerun)."
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| bf16 native:", torch.cuda.get_device_capability(0) >= (8, 0))

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07

torch 2.14.1+cu130 | CUDA 13.0 | bf16 native: False


In [3]:
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
for sub in ("outputs", "checkpoints", "tables", "env_logs", "data"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

PROJECT = "/content/project"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
if CODE_SOURCE == "github":
    url = GITHUB_REPO
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass  # public repo: no token needed
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, url, "/content/repo"], check=True)
    shutil.copytree(os.path.join("/content/repo", REPO_SUBDIR), PROJECT,
                    ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
else:
    shutil.copytree(DRIVE_CODE, PROJECT, ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
print("project code at", PROJECT, "->", sorted(os.listdir(PROJECT)))

Mounted at /content/drive
project code at /content/project -> ['.git', '.gitignore', '2606.04299v1.pdf', 'PROJECT_PLAN.md', 'README.md', 'configs', 'data', 'docs', 'env', 'notebooks', 'patches', 'pyproject.toml', 'reports', 'scripts', 'src', 'tests']


In [4]:
SID_ROOT = "/content/efficient-SID-code"
if not os.path.exists(SID_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/haojunqiu/efficient-SID-code", SID_ROOT], check=True)
subprocess.run(["git", "-C", SID_ROOT, "checkout", "-q", SID_COMMIT], check=True)
!pip install -q -r {PROJECT}/env/requirements-colab.txt
!pip install -q --no-deps -e {SID_ROOT}
!pip install -q -e {PROJECT}
os.environ["SID_ROOT"] = SID_ROOT
# editable installs are only picked up by *new* processes; make them importable in this kernel too
for p in (SID_ROOT, f"{PROJECT}/src"):
    if p not in sys.path:
        sys.path.insert(0, p)
import efficient_sid, sidx
print("efficient_sid OK | sidx", sidx.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for efficient-sid (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sidx (pyproject.toml) ... done
efficient_sid OK | sidx 0.1.0


## D3 — what precision does the "float32 bottleneck" really use?
The Table-1 preset keeps tensors in bf16 and wraps the kernel in `torch.autocast("cuda", dtype=float32)`.
(See docs/technical_notes.md §4, D3.)

In [5]:
import torch, torch.nn.functional as F
x = torch.randn(8, 675, device="cuda", dtype=torch.bfloat16)
y = torch.randn(16, 675, device="cuda", dtype=torch.bfloat16)
a = torch.tensor(0.5, device="cuda")                       # 0-dim fp32, like scheduler.alphas[t]
with torch.autocast(device_type="cuda", dtype=torch.float32):
    ay = a * y                                             # elementwise
    logits = x @ ay.T                                      # matmul
    w = F.softmax(logits, dim=1)
    out = torch.einsum("bn,nd->bd", w, y)
print("alpha*y:", ay.dtype, "| matmul:", logits.dtype, "| softmax:", w.dtype, "| einsum:", out.dtype)

alpha*y: torch.bfloat16 | matmul: torch.float32 | softmax: torch.float32 | einsum: torch.float32


## Timing calibration (resumable; outputs → Drive)

In [6]:
JOBS = f"{PROJECT}/configs/jobs/phase2_t4_timing.yaml"
OUT  = f"{DRIVE_ROOT}/outputs"
!python {PROJECT}/scripts/run_queue.py {JOBS} --out-root {OUT} --var SID_ROOT={SID_ROOT} --dry-run
!python {PROJECT}/scripts/run_queue.py {JOBS} --out-root {OUT} --var SID_ROOT={SID_ROOT} --budget-min 60

paper_fp32_native              sid_official     hash=0bc79975 images=1 seeds=3 pending=0
paper_shipped_bf16_native      sid_official     hash=124e1adf images=1 seeds=3 pending=0
flash_fp32_native              sid_official     hash=cc7af402 images=1 seeds=3 pending=0
paper_fp32_256sq_chunk4        sid_official     hash=3886b300 images=1 seeds=3 pending=0
flash_fp32_256sq               sid_official     hash=1485acfd images=1 seeds=3 pending=0
paper_fp32_128_3scales         sid_official     hash=d202505b images=1 seeds=3 pending=0
[runner] done=0 skipped(existing)=18 failed=0 remaining=0 stop=completed elapsed=6.22s
{
  "done": 0,
  "skipped_existing": 18,
  "failed": 0,
  "remaining": 0,
  "stopped_reason": "completed",
  "elapsed_s": 6.22,
  "per_job": {
    "paper_fp32_native": {
      "done": 0,
      "skipped": 3,
      "failed": 0,
      "remaining": 0
    },
    "paper_shipped_bf16_native": {
      "done": 0,
      "skipped": 3,
      "failed": 0,
      "remaining": 0
    },
    "f

In [7]:
import json, glob, pandas as pd
rows = []
for jsonl in glob.glob(f"{OUT}/phase2_t4_timing/*/*/samples.jsonl"):
    job = jsonl.split("/")[-3].split("__")[0]
    setup = json.load(open(jsonl.replace("samples.jsonl", "setup.json")))["setup"]
    recs = [json.loads(l) for l in open(jsonl)]
    secs = [r["seconds"] for r in recs]
    rows.append(dict(job=job, input_hw=setup["input_hw"], n_patches_finest=setup["num_patches_per_scale"][0],
                     setup_s=setup["setup_seconds"], samples=len(secs),
                     sec_per_sample_median=round(sorted(secs)[len(secs)//2], 2),
                     sec_first_sample=round(secs[0], 2), peak_vram_gb=max(r["peak_vram_gb"] or 0 for r in recs),
                     gpu=recs[0]["gpu"]))
df = pd.DataFrame(rows).sort_values("job")
display(df)
df.to_csv(f"{DRIVE_ROOT}/tables/phase2_t4_timing.csv", index=False)

,job,input_hw,n_patches_finest,setup_s,samples,sec_per_sample_median,sec_first_sample,peak_vram_gb,gpu
4,flash_fp32_256sq,"[256, 256]",58564,0.056,3,48.88,48.55,0.972,Tesla T4
2,flash_fp32_native,"[186, 248]",40248,0.037,3,22.11,22.66,0.667,Tesla T4
5,paper_fp32_128_3scales,"[128, 171]",17898,0.024,3,2.87,2.87,2.542,Tesla T4
3,paper_fp32_256sq_chunk4,"[256, 256]",58564,0.035,3,31.32,31.34,10.186,Tesla T4
0,paper_fp32_native,"[186, 248]",40248,0.154,3,13.49,13.43,9.444,Tesla T4
1,paper_shipped_bf16_native,"[186, 248]",40248,0.032,3,13.44,13.27,9.278,Tesla T4


### D2 — does the shipped bf16 mode change the output?
Same seed, fp32 vs as-shipped. Mean absolute difference in grey levels (0–255).

In [8]:
import numpy as np
from PIL import Image
def first(job, seed=10):
    return np.asarray(Image.open(glob.glob(f"{OUT}/phase2_t4_timing/{job}__*/balloons/seed_{seed:04d}.png")[0]), np.float32)
for seed in (10, 11, 12):
    try:
        a, b = first("paper_fp32_native", seed), first("paper_shipped_bf16_native", seed)
        c = first("flash_fp32_native", seed)
        print(f"seed {seed}: |fp32 - bf16 shipped| = {np.abs(a-b).mean():.2f}   |exact - flash (fp32)| = {np.abs(a-c).mean():.3f}")
    except IndexError:
        print(f"seed {seed}: not generated yet")

seed 10: |fp32 - bf16 shipped| = 0.13   |exact - flash (fp32)| = 0.001
seed 11: |fp32 - bf16 shipped| = 0.25   |exact - flash (fp32)| = 0.003
seed 12: |fp32 - bf16 shipped| = 0.16   |exact - flash (fp32)| = 0.003


## Optional: faiss (ablation AB6 only)

In [9]:
!pip install -q faiss-gpu-cu12
try:
    import faiss; print("faiss", faiss.__version__, "| GPUs:", faiss.get_num_gpus())
except Exception as e:
    print("faiss NOT usable:", repr(e), "-> AB6 stays optional")

faiss 1.14.1 | GPUs: 1


In [10]:
import datetime
stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
!pip freeze > {DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt
print("saved", f"{DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt")

saved /content/drive/MyDrive/CS787/env_logs/2026-11-15_1036_00_setup_smoke_freeze.txt


**Done when:** the timing table exists on Drive, D2/D3 are answered, and a rerun of the runner cell
prints `done=0 skipped(existing)=18`. Copy the numbers into PROJECT_PLAN §5.3 and docs/technical_notes.md.